# Človek-v-zanki: Vrata pred dejanjem, razvrščanje tveganj in revizijsko beleženje

README za to lekcijo uvaja človeka v zanki s kratkim odlomkom, ki uporabnika vpraša, naj `ODOBRI` ali `ZAVRNE` potem, ko je agent že dal odgovor. Ta vzorec je dober izhodiščni trenutek, a izdelki HITL običajno potrebujejo še tri dodatne elemente:

1. **vrata pred dejanjem**, ki se izvajajo **pred** izvedbo tveganega koraka agenta, da so stroški, nepopravljivost in zakasnitev pod nadzorom.
2. **razvrščanje tveganj**, tako da se nizkotveganjska dejanja izvajajo samodejno, srednje tvegana dejanja se odobrijo v serijah, visoko tvegana dejanja pa zahtevajo človeško potrditev.
3. **revizijski zapisnik in zanka popravljanja**, tako da je vsaka odločitev vrat zabeležena kot JSONL, zavrnitev pa ponovno sproži agenta s strukturiranim razlogom namesto zgolj prikaza `Popravljanje...`.

Ta zvezek gradi vsak od teh na istem osnovnem naboru kot `06-system-message-framework.ipynb`. Teče od začetka do konca v načinu `DEMO_MODE = True` (ni potrebnega interaktivnega vnosa) ali z realnimi pozivi `input()`, ko je `DEMO_MODE = False`. Opomba: v DEMO_MODE je ponovni poskus tretjega cilja skriptiran, zato so mehanizmi zanke vidni od začetka do konca. Resnična rekategorizacija, ki jo poganjajo popravki, zahteva `DEMO_MODE = False` in operaterja.

**Izven obsega (obravnavano v drugih lekcijah):** overjanje in nadzor dostopa (Lekcija 06 README grožnja 2), vmesnik za klice orodij (Lekcija 14 MAF poglobljeno), vzorci večagentnih razprav.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Vzorec 1: Vrata pred dejanjem

Odsek HITL v README-ju najprej kliče agenta, nato pa uporabnika prosi za odobritev izhoda. To je tok **po dejanju**. Agent je že izvedel, strošek klica LLM je že plačan, in vsak stranski učinek (poslan e-poštni naslov, zapisana vrstica v bazi, objavljen komentar) se je že zgodil.

Tok **pred dejanjem** vstavi vrata pred tem, ko agent izvede tvegani korak. Agent predlaga dejanje, vrata odločijo, ali ga izvesti, in stranski učinek se zgodi le ob odobritvi.

| Vidik | Odobritev po dejanju (odsek README) | Vrata pred dejanjem (ta zvezek) |
|---|---|---|
| Kdaj se izvede odobritev? | Po tem, ko agent ustvari izhod | Pred vsakim izvajanjem stranskega učinka |
| Strošek LLM ob zavrnitvi | Že plačan | Plača se le za predlog, ne za dejanje |
| Nepovratni stranski učinki | Možni (dejanje se je že zgodilo) | Preprečeni |
| Jasnost revizije | Odobritev je izpisna izjava | Odobritev je JSON zapis s časovnim žigom, dejanjem, razlogom |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Vzorec 2: Razvrščanje tveganja

Ne vsako dejanje zahteva človeško odobritev. Preverjanje samo za branje preko javnega API-ja ima drugačna tveganja kot pošiljanje e-pošte stranki. Enako obravnavanje obojega povzroča nepotrebno pozornost operaterja in upočasni agenta.

Enostaven model s tremi nivoji:

| Nivo | Primeri | Potek odobritve |
|---|---|---|
| `nizek` (samo za branje) | Iskanje v bazi znanja, pregled letalskih možnosti, pridobivanje javne spletne strani | Samodejno izvajanje, zabeleženo za revizijo |
| `srednji` (cenejša mutacija) | Predpomnjenje rezultata, povečanje števca, načrtovanje opomnika | Samodejno izvajanje, a dnevni pregled v skupinah |
| `visok` (zunanja interakcija ali nepopravljivo) | Pošiljanje e-pošte, obremenitev kartice, objava v javnem kanalu | Blokada do človeške odobritve |

To je en sam način razvrščanja. Producentni sistemi pogosto uporabljajo bolj podrobne nivoje (npr. AWS IAM dovolilni nivoji, nivoji dostopa na osnovi vlog). Verzija s tremi nivoji spodaj je najmanjša uporabna verzija za agenta, ki meša dejanja samo za branje in tista s stranskimi učinki.

Razvrščevalnik spodaj uporablja ključne besede kot heuristiko, da demo ostane determinističen in ugoden. V produkcijskem sistemu bi zamenjali to z učenim razvrščevalcem ali sistemom pravil.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Vzorec 3: Revizijska dnevniška in zanka revizije

`print("Odgovor odobren.")` ni revizijski dnevnik. Za zaupanje naj bo vsaka odločitev vrat zabeležena kot strukturiran dogodek, ki ga lahko pozneje poizvedujete, ponovno predvajate ali priložite pregledu incidenta.

Dve sestavini:

1. **Samo dodajanje JSONL.** Ena vrstica na odločitev, s časovnim žigom, akcijo, nivojem, odločitvijo, razlogom. Enostavno za iskanje, enostavno za pošiljanje v resničen dnevnik pozneje.
2. **Zanka revizije ob zavrnitvi.** Ko vrata vrnejo `deny`, si agent sam ponovno zastavi vprašanje z razlogom zavrnitve v kontekstu, da lahko naslednji predlog prepreči težavo.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Dodatni viri

Več drugih javnih projektov implementira različice teh HITL vzorcev. Primerjajte pristope, da boste našli tistega, ki ustreza vaši tehnologiji:

- **LangChain** ovijalci orodij human-in-the-loop ([dokumentacija](https://python.langchain.com/docs/integrations/tools/human_tools)): ovijalci orodij, ki zaustavijo izvajanje za človeški vnos.
- **AutoGen** `UserProxyAgent` ([dokumentacija v0.2](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ je to preuredil): uporablja vlogo agenta posebej za predstavitev človeka v večagentnih pogovorih.
- **Microsoft Agent Framework (MAF)** middleware za klice funkcij ([dokumentacija](https://learn.microsoft.com/agent-framework/)): middleware, ki teče okoli vsakega klica orodja/funkcije, primeren za kontrolno logiko in poteke odobritve.

Vsak projekt drugače obravnava tri podvzorce: LangChain jih ovije kot orodja, AutoGen uporablja vlogo agenta, Microsoft Agent Framework pa middleware za klice funkcij. Preberite eno ali dve implementaciji do konca, preden izberete zasnovo za svojega agenta.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
